In [5]:
import pandas as pd
from collections import Counter

files = [
    "2019-Oct.csv",
    "2019-Nov.csv",
    "2019-Dec.csv",
    "2020-Jan.csv"
]

chunksize = 500_000
usecols = ["event_time", "event_type", "category_code", "brand"]

for file_path in files:
    print("=" * 100)
    print(f"파일명: {file_path}")

    total_rows = 0
    event_type_counts = Counter()
    category_counts = Counter()
    brand_counts = Counter()
    missing_counts = Counter()

    min_time = None
    max_time = None
    columns = None
    dtypes = None

    for chunk in pd.read_csv(file_path, usecols=usecols, chunksize=chunksize, low_memory=True):
        if columns is None:
            columns = chunk.columns.tolist()
            dtypes = chunk.dtypes.astype(str)

        total_rows += len(chunk)
        missing_counts.update(chunk.isna().sum().to_dict())

        event_type_counts.update(
            chunk["event_type"].dropna().astype(str).value_counts().to_dict()
        )
        category_counts.update(
            chunk["category_code"].dropna().astype(str).value_counts().to_dict()
        )
        brand_counts.update(
            chunk["brand"].dropna().astype(str).value_counts().to_dict()
        )

        times = pd.to_datetime(chunk["event_time"], errors="coerce")
        chunk_min = times.min()
        chunk_max = times.max()

        if pd.notna(chunk_min):
            min_time = chunk_min if min_time is None else min(min_time, chunk_min)
        if pd.notna(chunk_max):
            max_time = chunk_max if max_time is None else max(max_time, chunk_max)

    print("\n[1] 컬럼명")
    print(columns)

    print("\n[2] dtype")
    print(dtypes)

    print(f"\n[3] 총 행 수: {total_rows:,}")
    print(f"[4] 시간 범위: {min_time} ~ {max_time}")

    print("\n[5] 결측치 개수")
    for col, cnt in missing_counts.items():
        print(f"{col}: {cnt:,}")

    print("\n[6] event_type 종류 / 개수")
    for event, cnt in event_type_counts.most_common():
        print(f"{event}: {cnt:,}")

    print("\n[7] category_code 상위 30개")
    for cat, cnt in category_counts.most_common(30):
        print(f"{cat}: {cnt:,}")

    print("\n[8] brand 상위 30개")
    for brand, cnt in brand_counts.most_common(30):
        print(f"{brand}: {cnt:,}")

파일명: 2019-Oct.csv

[1] 컬럼명
['event_time', 'event_type', 'category_code', 'brand']

[2] dtype
event_time       str
event_type       str
category_code    str
brand            str
dtype: str

[3] 총 행 수: 42,448,764
[4] 시간 범위: 2019-10-01 00:00:00+00:00 ~ 2019-10-31 23:59:59+00:00

[5] 결측치 개수
event_time: 0
event_type: 0
category_code: 13,515,609
brand: 6,117,080

[6] event_type 종류 / 개수
view: 40,779,399
cart: 926,516
purchase: 742,849

[7] category_code 상위 30개
electronics.smartphone: 11,507,231
electronics.clocks: 1,311,033
computers.notebook: 1,137,623
electronics.video.tv: 1,113,750
electronics.audio.headphone: 1,100,188
appliances.kitchen.refrigerators: 887,755
appliances.kitchen.washer: 869,404
appliances.environment.vacuum: 801,670
apparel.shoes: 763,901
auto.accessories.player: 470,208
computers.desktop: 424,242
apparel.shoes.keds: 410,304
furniture.bedroom.bed: 358,453
electronics.tablet: 316,735
electronics.audio.subwoofer: 313,664
furniture.living_room.cabinet: 301,410
construction.t

In [9]:
import pandas as pd
from collections import Counter

chunksize = 500_000

targets = [
    ("2019-Nov.csv", "electronics.smartphone"),
    ("2019-Nov.csv", "construction.tools.light"),
    ("2019-Dec.csv", "electronics.smartphone"),
    ("2019-Dec.csv", "construction.tools.light"),
    ("2020-Jan.csv", "electronics.smartphone"),
    ("2020-Jan.csv", "construction.tools.light")
]

for file_path, category in targets:
    brand_counts = Counter()

    for chunk in pd.read_csv(
        file_path,
        usecols=["category_code", "brand"],
        chunksize=chunksize
    ):
        filtered = chunk[
            (chunk["category_code"] == category) &
            (chunk["brand"].notna())
        ]
        brand_counts.update(filtered["brand"].astype(str).value_counts().to_dict())

    print("=" * 80)
    print(file_path, "-", category)
    for brand, cnt in brand_counts.most_common(10):
        print(f"{brand}: {cnt:,}")

2019-Nov.csv - electronics.smartphone
samsung: 5,316,962
apple: 4,658,729
xiaomi: 3,331,784
huawei: 1,237,930
oppo: 811,698
meizu: 201,703
vivo: 159,327
oneplus: 110,173
honor: 107,840
nokia: 94,164
2019-Nov.csv - construction.tools.light
xiaomi: 4,045
mantra: 2,956
philips: 2,013
ansmann: 1,615
puckator: 1,328
eco: 1,239
rombica: 788
deluxe: 641
stanley: 616
ultraflash: 568
2019-Dec.csv - electronics.smartphone
samsung: 252,372
xiaomi: 198,534
apple: 131,510
huawei: 48,560
nokia: 30,906
sony: 28,073
meizu: 23,678
vivo: 21,762
oppo: 21,442
lg: 20,069
2019-Dec.csv - construction.tools.light
samsung: 5,503,879
apple: 4,005,910
xiaomi: 3,509,434
huawei: 1,565,303
oppo: 746,041
meizu: 189,216
vivo: 174,231
honor: 120,363
nokia: 75,297
oneplus: 59,549
2020-Jan.csv - electronics.smartphone
samsung: 152,469
xiaomi: 88,730
apple: 72,006
huawei: 34,915
nokia: 16,806
sony: 12,702
oppo: 11,808
meizu: 8,116
lg: 7,950
vivo: 6,935
2020-Jan.csv - construction.tools.light
samsung: 4,956,998
apple: 4,0

In [11]:
nov_file = "2019-Nov.csv"
dec_file = "2019-Dec.csv"

smartphone_ids = set()
light_ids = set()

# 11월 smartphone product_id 수집
for chunk in pd.read_csv(
    nov_file,
    usecols=["product_id", "category_code"],
    chunksize=chunksize,
    low_memory=True
):
    filtered = chunk[chunk["category_code"] == "electronics.smartphone"]
    smartphone_ids.update(filtered["product_id"].dropna().astype(str))

# 12월 construction.tools.light product_id 수집
for chunk in pd.read_csv(
    dec_file,
    usecols=["product_id", "category_code"],
    chunksize=chunksize,
    low_memory=True
):
    filtered = chunk[chunk["category_code"] == "construction.tools.light"]
    light_ids.update(filtered["product_id"].dropna().astype(str))

overlap_ids = smartphone_ids & light_ids

print("11월 smartphone product_id 수:", len(smartphone_ids))
print("12월 construction.tools.light product_id 수:", len(light_ids))
print("겹치는 product_id 수:", len(overlap_ids))
print("겹치는 product_id 샘플:", list(overlap_ids)[:20])

11월 smartphone product_id 수: 1341
12월 construction.tools.light product_id 수: 2021
겹치는 product_id 수: 724
겹치는 product_id 샘플: ['1004739', '1005229', '1004361', '1004751', '1004966', '1005017', '1005143', '1004375', '1004461', '1004300', '1002827', '1004197', '1003311', '1005070', '1004503', '1004093', '1004794', '1005284', '1004865', '1005008']


In [12]:
nov_light_ids = set()
dec_light_ids = set()

# 11월 light
for chunk in pd.read_csv(
    "2019-Nov.csv",
    usecols=["product_id", "category_code"],
    chunksize=chunksize,
    low_memory=True
):
    filtered = chunk[chunk["category_code"] == "construction.tools.light"]
    nov_light_ids.update(filtered["product_id"].dropna().astype(str))

# 12월 light
for chunk in pd.read_csv(
    "2019-Dec.csv",
    usecols=["product_id", "category_code"],
    chunksize=chunksize,
    low_memory=True
):
    filtered = chunk[chunk["category_code"] == "construction.tools.light"]
    dec_light_ids.update(filtered["product_id"].dropna().astype(str))

light_overlap = nov_light_ids & dec_light_ids

print("11월 light product_id 수:", len(nov_light_ids))
print("12월 light product_id 수:", len(dec_light_ids))
print("겹치는 light product_id 수:", len(light_overlap))

if len(nov_light_ids) > 0:
    print("11월 light 기준 유지 비율:", round(len(light_overlap) / len(nov_light_ids) * 100, 2), "%")
if len(dec_light_ids) > 0:
    print("12월 light 기준 기존 light 비율:", round(len(light_overlap) / len(dec_light_ids) * 100, 2), "%")

11월 light product_id 수: 231
12월 light product_id 수: 2021
겹치는 light product_id 수: 45
11월 light 기준 유지 비율: 19.48 %
12월 light 기준 기존 light 비율: 2.23 %
